# 07 评论 NLP 分析

对用户评论进行文本挖掘：
- jieba 中文分词
- 停用词过滤
- 低评分/高评分词频对比
- 词云可视化
- 评论维度归因分析（物流/品类/地区/金额）

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from utils.db_connector import get_engine
engine = get_engine()

import jieba
print("jieba 分词库加载完成")

## 7.1 加载评论数据

In [ ]:
# 加载评论相关数据
query = """
SELECT 
    r.review_id, r.order_id, r.review_score, r.review_comment_message,
    o.order_purchase_timestamp, o.order_delivered_customer_date,
    o.order_estimated_delivery_date, o.customer_id,
    i.product_id, i.price,
    p.product_category_name, c.customer_state
FROM olist_order_reviews r
LEFT JOIN olist_orders o ON r.order_id = o.order_id
LEFT JOIN olist_order_items i ON r.order_id = i.order_id
LEFT JOIN olist_products p ON i.product_id = p.product_id
LEFT JOIN olist_customers c ON o.customer_id = c.customer_id
"""

with engine.connect() as conn:
    df = pd.read_sql_query(query, conn)

# 计算配送天数
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['is_delayed'] = (df['delay_days'] > 0).astype(int)

# 定义评分分组
df['score_group'] = df['review_score'].apply(
    lambda x: 'low' if x <= 2 else ('high' if x >= 4 else 'mid')
)

print(f"评论总数: {len(df)}")
print(f"低评分(≤2): {(df['score_group']=='low').sum()}")
print(f"高评分(≥4): {(df['score_group']=='high').sum()}")
df.head()

## 7.2 评分维度分析

分析低评分订单的归因：物流、品类、地区、金额等维度。

In [ ]:
# 多维度分析
low_df = df[df['score_group'] == 'low']
high_df = df[df['score_group'] == 'high']

print("=== 物流维度 ===")
logistics_compare = pd.DataFrame({
    '低评分': [low_df['delivery_days'].mean(), low_df['is_delayed'].mean(),
              low_df.loc[low_df['is_delayed']==1, 'delay_days'].mean()],
    '高评分': [high_df['delivery_days'].mean(), high_df['is_delayed'].mean(),
              high_df.loc[high_df['is_delayed']==1, 'delay_days'].mean()]
}, index=['平均配送天数', '延迟率', '平均延迟天数'])
print(logistics_compare)

print("\n=== 品类维度(低评分率 Top 5) ===")
category_low_rate = df.groupby('product_category_name').apply(
    lambda x: (x['score_group'] == 'low').mean()
).sort_values(ascending=False)
print(category_low_rate.head(5))

print("\n=== 地区维度(低评分率 Top 5) ===")
state_low_rate = df.groupby('customer_state').apply(
    lambda x: (x['score_group'] == 'low').mean()
).sort_values(ascending=False)
print(state_low_rate.head(5))

print(f"\n=== 金额维度 ===")
print(f"低评分客单价均值: ¥{low_df['price'].mean():.2f}")
print(f"高评分客单价均值: ¥{high_df['price'].mean():.2f}")

## 7.3 评分维度可视化

In [ ]:
# 评分分析可视化
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 评分分布
score_counts = df['review_score'].value_counts().sort_index()
colors = ['#e74c3c', '#e74c3c', '#f39c12', '#2ecc71', '#2ecc71']
axes[0, 0].bar(score_counts.index, score_counts.values, color=colors, alpha=0.8)
axes[0, 0].set_xlabel('评分')
axes[0, 0].set_ylabel('评论数')
axes[0, 0].set_title('评分分布')

# 品类低评分 Top 10
top10_cats = category_low_rate.head(10)
axes[0, 1].barh(range(len(top10_cats)), top10_cats.values, color='#e74c3c', alpha=0.7)
axes[0, 1].set_yticks(range(len(top10_cats)))
axes[0, 1].set_yticklabels(top10_cats.index, fontsize=8)
axes[0, 1].set_xlabel('低评分占比')
axes[0, 1].set_title('品类低评分率 Top 10')

# 地区低评分 Top 10
top10_states = state_low_rate.head(10)
axes[1, 0].barh(range(len(top10_states)), top10_states.values, color='#3498db', alpha=0.7)
axes[1, 0].set_yticks(range(len(top10_states)))
axes[1, 0].set_yticklabels(top10_states.index)
axes[1, 0].set_xlabel('低评分占比')
axes[1, 0].set_title('各州低评分率 Top 10')

# 归因对比热力图
heatmap_data = pd.DataFrame({
    '配送天数': [low_df['delivery_days'].mean(), high_df['delivery_days'].mean()],
    '延迟率': [low_df['is_delayed'].mean(), high_df['is_delayed'].mean()],
    '客单价': [low_df['price'].mean(), high_df['price'].mean()],
}, index=['低评分', '高评分'])
heatmap_norm = heatmap_data.apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))
im = axes[1, 1].imshow(heatmap_norm.values, cmap='RdYlGn_r', aspect='auto')
axes[1, 1].set_xticks(range(len(heatmap_data.columns)))
axes[1, 1].set_xticklabels(heatmap_data.columns)
axes[1, 1].set_yticks(range(len(heatmap_data.index)))
axes[1, 1].set_yticklabels(heatmap_data.index)
axes[1, 1].set_title('低评分 vs 高评分 归因对比')
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        axes[1, 1].text(j, i, f'{heatmap_data.iloc[i, j]:.2f}', ha='center', va='center', fontsize=11)

plt.tight_layout()
plt.show()

## 7.4 jieba 分词与词频统计

In [ ]:
# 中文停用词
zh_stopwords = set([
    '的', '了', '是', '我', '很', '在', '也', '都', '就', '不',
    '有', '和', '那', '这', '个', '一', '上', '为', '什么', '么',
    '吗', '呢', '啊', '哦', '嗯', '还', '而', '但', '又', '或',
    '会', '能', '可以', '把', '被', '让', '给', '到', '着', '过',
    '吧', '呀', '哈', '嘿', '哎', '嘛', '噢', '喔', '哇',
    '比较', '已经', '真的', '觉得', '知道', '应该', '可能', '需要',
    '因为', '所以', '如果', '虽然', '但是', '而且', '然后', '不过',
    '这个', '那个', '这些', '那些', '一个', '一些', '一下', '一点',
    '没', '要', '去', '做', '说', '看', '想', '买',
])

# 筛选有评论文本的数据
low_text = df[(df['score_group'] == 'low') &
              (df['review_comment_message'].notna()) &
              (df['review_comment_message'].str.strip() != '')]['review_comment_message']

high_text = df[(df['score_group'] == 'high') &
               (df['review_comment_message'].notna()) &
               (df['review_comment_message'].str.strip() != '')]['review_comment_message']

print(f"低评分有效评论数: {len(low_text)}")
print(f"高评分有效评论数: {len(high_text)}")

def preprocess_text(texts):
    all_words = []
    for text in texts:
        text = str(text).strip()
        text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', text)
        words = list(jieba.cut(text))
        words = [w for w in words if w not in zh_stopwords and len(w) > 1]
        all_words.extend(words)
    return all_words

low_words = preprocess_text(low_text)
low_freq = Counter(low_words).most_common(50)

high_words = preprocess_text(high_text)
high_freq = Counter(high_words).most_common(50)

print(f"\n低评分高频词 Top 20:")
for word, count in low_freq[:20]:
    print(f"  {word}: {count}")

## 7.5 词云可视化

In [ ]:
# 词云生成
try:
    from wordcloud import WordCloud
    
    low_word_dict = dict(low_freq)
    wc = WordCloud(
        width=800, height=400,
        background_color='white',
        max_words=100,
        colormap='Reds',
        font_path='C:/Windows/Fonts/simhei.ttf'
    ).generate_from_frequencies(low_word_dict)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('低评分评论词云', fontsize=14)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("wordcloud 未安装，跳过词云生成。安装命令: pip install wordcloud")

## 7.6 高频词对比

In [ ]:
# 高频词对比表
low_freq_df = pd.DataFrame(low_freq, columns=['word', 'low_score_count'])
high_freq_df = pd.DataFrame(high_freq, columns=['word', 'high_score_count'])
compare_df = low_freq_df.merge(high_freq_df, on='word', how='outer').fillna(0)
compare_df['low_score_count'] = compare_df['low_score_count'].astype(int)
compare_df['high_score_count'] = compare_df['high_score_count'].astype(int)
compare_df['diff_ratio'] = (compare_df['low_score_count'] - compare_df['high_score_count']) / \
                            (compare_df['low_score_count'] + compare_df['high_score_count'] + 1)
compare_df = compare_df.sort_values('diff_ratio', ascending=False)

print("低评分特有高频词 Top 10:")
compare_df.head(10)

## 小结

- 低评分订单的物流延迟率显著高于高评分订单
- 某些品类（如电子产品、家具）低评分率较高
- 文本分析显示低评分评论中「延迟」「质量」「退货」等词频较高
- 建议重点优化物流时效和高差评品类的品控